# Second Improvement: YOLOv8 + CLIP Linear Probe (Cascaded Verification)

**Course:** CS437/CS5317/EE414/EE513 — Deep Learning, Spring 2026
**Group 4:** Muhammad Baqir Hassan Babar (27100340) & Momin Fahed Khan (27100082)

**Project:** Reducing False Positives in Real-Time Forest Fire and Smoke Detection via Semantic Verification

---

## What this notebook does

Improvement 1 used CLIP **zero-shot** as a post-hoc verifier — text prompts were the prior. The headline win was a 77.6% relative drop in hard-negative FPR (2.44% → 0.55%), at the cost of −16.5 pp mean recall.

This notebook keeps the **same cascaded architecture** as I1 (YOLO → if `conf ≥ τ_high` keep; else verify) and **the same 20% crop margin**, but replaces the text-prompt classifier with a **supervised linear probe trained on CLIP image features extracted from D-Fire crops**. The probe is logistic regression over CLIP ViT-B/32's 512-d image embeddings, predicting one of three classes: `smoke`, `fire`, `neg`. A 1-layer MLP variant is included as an ablation.

**Hypothesis:** a learned forest-domain decision boundary in CLIP feature space recovers some of I1's recall loss while preserving the FPR gain — because zero-shot CLIP's "is this fire?" prior is web-scale generic, not D-Fire-specific. I1's prompt-bank ablation already showed the 12 confounder prompts do nearly all the work; replacing them with a learned negative cluster should be at worst comparable and at best better.

## What you must upload to Kaggle before running

1. **D-Fire dataset** — attach the public Kaggle dataset `sayedgamal99/smoke-fire-detection-yolo` (same as `baseline.ipynb` and `improvement1_final.ipynb`). Mounts at `/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data`.
2. **Baseline YOLO weights** — upload `./yoloonly_best_model/yolov8n_dfire_best.pt` from this repo as a private Kaggle Dataset (re-use the existing `mominfahed/baseline` slug if available). It must be reachable at `/kaggle/input/datasets/mominfahed/baseline/yolov8n_dfire_best.pt`. If your slug differs, edit `CONFIG["baseline_weights_input"]` in the config cell.

GPU: **T4 or P100** (matches I1 hardware so latency is comparable).
Expected wall time: ~30–55 min total (most of it is the full-test-set eval, run twice to re-baseline I1 within the same harness).

## Pipeline summary

```
D-Fire test image
        ↓
YOLOv8n detection (frozen, weights from baseline.ipynb)
        ↓
For each detection:
    if yolo_conf >= τ_high (0.70):  auto-keep (skip verifier)
    else:
        crop with 20% margin
        encode with frozen CLIP ViT-B/32  (same as I1)
        classify with LEARNED LINEAR PROBE  ← NEW vs Improvement 1
        keep iff probe argmax == YOLO class
```

The cascade scaffold and CLIP encoder are byte-identical to I1. The *only* change is what scores the CLIP features.


## 1. Configuration (single source of truth)

All hyperparameters live here. Edit only this cell to change behavior.

In [1]:
# === Improvement 2 Configuration ===
CONFIG = {
    # Reproducibility
    "seed": 42,

    # Data paths (Kaggle)
    "dfire_root":             "/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data",
    "baseline_weights_input": "/kaggle/input/datasets/mominfahed/baseline/yolov8n_dfire_best.pt",
    "work_dir":               "/kaggle/working",

    # YOLO inference (must match Improvement 1 exactly for comparability)
    "yolo_conf":   0.25,
    "yolo_iou":    0.45,
    "imgsz":       640,
    "tau_high":    0.70,
    "crop_margin": 0.20,

    # CLIP
    "clip_model":  "ViT-B/32",

    # Probe training data
    "mining_conf":           0.10,    # YOLO conf used to mine FP crops on train negatives
    "max_random_negatives":  5000,    # cap on random crops added to negative set
    "max_mined_negatives":   None,    # None = use all mined FPs
    "max_positives_per_cls": None,    # None = use all positive GT crops

    # Logistic regression probe (default)
    "lr_C":         1.0,
    "lr_max_iter":  2000,

    # MLP probe (ablation)
    "mlp_hidden":     64,
    "mlp_epochs":     40,
    "mlp_batch_size": 1024,
    "mlp_lr":         1e-3,
    "mlp_dropout":    0.2,

    # Eval
    "max_eval_images": None,   # set to e.g. 500 for a smoke test; None = full 4306-image test set
    "tau_grid":        [0.00, 0.30, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00],
    "margin_grid":     [0.00, 0.20, 0.40, 0.60],

    # Outputs
    "metrics_path":  "/kaggle/working/improvement2_metrics.json",
    "probe_lr_path": "/kaggle/working/probe_lr.pkl",
    "probe_mlp_path":"/kaggle/working/probe_mlp.pt",
}

# Probe class space — must match the cascade verify_detection logic.
# YOLO emits class 0=smoke, 1=fire. Probe emits 0=smoke, 1=fire, 2=neg.
PROBE_CLASS_NAMES = ["smoke", "fire", "neg"]
CLASS_NAMES = ["smoke", "fire"]   # YOLO classes (matches I1)


## 2. Environment, imports, reproducibility

In [2]:
import subprocess, sys

def pip_install(args):
  subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + args)

pip_install(["ftfy", "regex", "PyYAML", "scikit-learn"])
pip_install(["--no-deps", "git+https://github.com/openai/CLIP.git"])
pip_install(["--no-deps", "ultralytics"])
pip_install(["py-cpuinfo", "thop", "seaborn", "pandas"])  # ultralytics' non-torch deps

import os, random, json, yaml, time, shutil, pickle
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import clip
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

SEED = CONFIG["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
CONFIG["device"] = device

print(f"Device       : {device}")
if device == 'cuda':
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch      : {torch.__version__}")
import ultralytics
print(f"Ultralytics  : {ultralytics.__version__}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.2 MB/s eta 0:00:00
Device       : cuda
GPU          : Tesla T4
VRAM         : 15.6 GB
PyTorch      : 2.10.0+cu128
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics  : 8.4.47


## 3. Data paths

We use the same D-Fire mount and the same YOLO weights as Improvement 1. The asserts below fail loudly if either Kaggle input is missing — fix the dataset attachment before continuing.

In [3]:
DFIRE_ROOT             = CONFIG["dfire_root"]
BASELINE_WEIGHTS_INPUT = CONFIG["baseline_weights_input"]
WORK_DIR               = CONFIG["work_dir"]
os.makedirs(WORK_DIR, exist_ok=True)

train_img = os.path.join(DFIRE_ROOT, "train", "images")
train_lbl = os.path.join(DFIRE_ROOT, "train", "labels")
val_img   = os.path.join(DFIRE_ROOT, "val",   "images")
val_lbl   = os.path.join(DFIRE_ROOT, "val",   "labels")
test_img  = os.path.join(DFIRE_ROOT, "test",  "images")
test_lbl  = os.path.join(DFIRE_ROOT, "test",  "labels")

all_ok = True
for label, d in [("train/images", train_img), ("train/labels", train_lbl),
                 ("val/images",   val_img),   ("val/labels",   val_lbl),
                 ("test/images",  test_img),  ("test/labels",  test_lbl)]:
    ok = os.path.isdir(d)
    print(f"{'[OK]' if ok else '[MISSING]'} {label}: {d}")
    all_ok &= ok

assert all_ok, ("D-Fire dataset not attached. Add Kaggle dataset "
                "sayedgamal99/smoke-fire-detection-yolo before running.")

assert os.path.exists(BASELINE_WEIGHTS_INPUT), (
    f"Baseline weights missing at {BASELINE_WEIGHTS_INPUT}. Upload "
    "yoloonly_best_model/yolov8n_dfire_best.pt as a Kaggle Dataset and attach it."
)

# Stage baseline weights to /kaggle/working
BASELINE_WEIGHTS = os.path.join(WORK_DIR, "yolov8n_dfire_best.pt")
if not os.path.exists(BASELINE_WEIGHTS):
    shutil.copy2(BASELINE_WEIGHTS_INPUT, BASELINE_WEIGHTS)
print(f"\n[OK] baseline weights staged to {BASELINE_WEIGHTS}")

# Dataset YAML (used by Ultralytics yolo.val for the YOLO-only reference run)
dataset_yaml_path = os.path.join(WORK_DIR, "dfire.yaml")
with open(dataset_yaml_path, "w") as f:
    yaml.dump({
        "path": DFIRE_ROOT, "train": "train/images", "val": "val/images",
        "test": "test/images", "nc": 2, "names": ["smoke", "fire"],
    }, f, default_flow_style=False, sort_keys=False)
print(f"[OK] wrote {dataset_yaml_path}")

# Quick split summary
for split, img_dir, lbl_dir in [("train", train_img, train_lbl),
                                 ("val",   val_img,   val_lbl),
                                 ("test",  test_img,  test_lbl)]:
    n_img = len([f for f in os.listdir(img_dir)
                 if f.lower().endswith(('.jpg','.jpeg','.png'))])
    n_lbl = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')])
    n_empty = sum(1 for f in os.listdir(lbl_dir)
                  if f.endswith('.txt') and os.path.getsize(os.path.join(lbl_dir, f)) == 0)
    print(f"{split:5s}  images={n_img:6d}  labels={n_lbl:6d}  negatives(empty)={n_empty:6d}")


[OK] train/images: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/train/images
[OK] train/labels: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/train/labels
[OK] val/images: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images
[OK] val/labels: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels
[OK] test/images: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/test/images
[OK] test/labels: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/test/labels

[OK] baseline weights staged to /kaggle/working/yolov8n_dfire_best.pt
[OK] wrote /kaggle/working/dfire.yaml
train  images= 14122  labels= 14122  negatives(empty)=  6458
val    images=  3099  labels=  3099  negatives(empty)=  1375
test   images=  4306  labels=  4306  negatives(empty)=  2005


## 4. Load YOLO + CLIP

Both are frozen. CLIP weights download from OpenAI on first call (cached in `~/.cache/clip`).

In [4]:
from ultralytics import YOLO

yolo = YOLO(BASELINE_WEIGHTS)
print(f"YOLO from    : {BASELINE_WEIGHTS}")
print(f"YOLO params  : {sum(p.numel() for p in yolo.model.parameters()):,}")
print(f"YOLO classes : {yolo.names}")

print(f"\nLoading CLIP {CONFIG['clip_model']} ...")
clip_model, clip_preprocess = clip.load(CONFIG["clip_model"], device=device, jit=False)
clip_model.eval()
for p in clip_model.parameters():
    p.requires_grad_(False)
print(f"CLIP params  : {sum(p.numel() for p in clip_model.parameters()):,}")

# Sanity: encoded feature dim (512 for ViT-B/32, 768 for ViT-L/14)
with torch.no_grad():
    _dummy = torch.zeros(1, 3, 224, 224, device=device)
    CLIP_FEAT_DIM = clip_model.encode_image(_dummy).shape[-1]
print(f"CLIP image feature dim: {CLIP_FEAT_DIM}")


YOLO from    : /kaggle/working/yolov8n_dfire_best.pt
YOLO params  : 3,011,238
YOLO classes : {0: 'smoke', 1: 'fire'}

Loading CLIP ViT-B/32 ...


100%|████████████████████████████████████████| 338M/338M [00:03<00:00, 104MiB/s]


CLIP params  : 151,277,313
CLIP image feature dim: 512


## 5. YOLO-only reference numbers (`yolo.val` — same call as `baseline.ipynb`)

These are produced by Ultralytics' native evaluator. They reproduce the baseline reference numbers (mAP50 ≈ 0.7614, mean P ≈ 0.7696, mean R ≈ 0.6971). They are **not** directly comparable to the YOLO-only numbers from our custom harness later (which uses 11-point AP and a simpler matcher) — see CLAUDE.md for the explanation. We report both. The custom-harness number is the apples-to-apples one against I1 and I2.

In [5]:
print("Running YOLO-only evaluation on the test split (Ultralytics native)...")
metrics = yolo.val(
    data    = dataset_yaml_path,
    split   = "test",
    imgsz   = CONFIG["imgsz"],
    batch   = 16,
    device  = device,
    verbose = True,
    plots   = False,
)
yolo_native = {
    "mAP50":         float(metrics.box.map50),
    "mAP50_95":      float(metrics.box.map),
    "mean_precision":float(metrics.box.mp),
    "mean_recall":   float(metrics.box.mr),
    "ap50_smoke":    float(metrics.box.ap50[0]),
    "ap50_fire":     float(metrics.box.ap50[1]),
}
print("\nYOLO-only (Ultralytics native, reference):")
for k, v in yolo_native.items():
    print(f"  {k:18s} = {v:.4f}")


Running YOLO-only evaluation on the test split (Ultralytics native)...
Ultralytics 8.4.47 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 1.1±1.4 ms, read: 7.2±4.2 MB/s, size: 91.1 KB)
val: Scanning /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/test/labels... 4295 images, 2005 backgrounds, 15 corrupt: 100% ━━━━━━━━━━━━ 4306/4306 203.3it/s 21.2s<0.1s
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/test/images/WEB10769.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0297]
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/test/images/WEB10775.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0156]
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/test/images/WEB11243.jpg: ignoring corrupt image/label:

## 6. Prompt bank for the Improvement 1 zero-shot re-baseline

Copied verbatim from `improvement1_final.ipynb` cell 14 so we can re-run I1 inside this notebook on the same harness as I2 — that gives us a fair, harness-controlled I1 vs I2 comparison without trusting cross-notebook numbers.

In [6]:
FIRE_PROMPTS = [
    "a photo of fire",
    "a photo of flames burning",
    "a photo of a forest fire",
    "a photo of a wildfire with visible flames",
    "a close-up photo of orange burning flames",
]

SMOKE_PROMPTS = [
    "a photo of smoke",
    "a photo of a smoke plume rising",
    "a photo of thick smoke from a fire",
    "a photo of grey smoke in a forest",
    "a photo of smoke from burning trees",
]

NEG_PROMPTS = [
    "a photo of fog",
    "a photo of mist over a forest",
    "a photo of clouds in the sky",
    "a photo of haze over mountains",
    "a photo of a cloudy sky",
    "a photo of sunlight shining through trees",
    "a photo of bright sun glare",
    "a photo of a lamp or streetlight",
    "a photo of a reflective surface",
    "a photo of a normal forest",
    "a photo of a clear landscape without fire",
    "a photo of a building or structure",
]

ALL_PROMPTS = FIRE_PROMPTS + SMOKE_PROMPTS + NEG_PROMPTS
N_FIRE, N_SMOKE, N_NEG = len(FIRE_PROMPTS), len(SMOKE_PROMPTS), len(NEG_PROMPTS)
FIRE_IDX  = list(range(0, N_FIRE))
SMOKE_IDX = list(range(N_FIRE, N_FIRE + N_SMOKE))
NEG_IDX   = list(range(N_FIRE + N_SMOKE, N_FIRE + N_SMOKE + N_NEG))

def bucket_of_text(idx):
    if idx in FIRE_IDX:  return "fire"
    if idx in SMOKE_IDX: return "smoke"
    return "neg"

@torch.no_grad()
def encode_prompts(model, prompt_list, device):
    tokens = clip.tokenize(prompt_list).to(device)
    feats  = model.encode_text(tokens)
    feats  = feats / feats.norm(dim=-1, keepdim=True)
    return feats

text_feats = encode_prompts(clip_model, ALL_PROMPTS, device)
print(f"Encoded {len(ALL_PROMPTS)} prompts -> shape {tuple(text_feats.shape)}")


Encoded 22 prompts -> shape (22, 512)


## 7. Probe training data

The probe is trained on **train-split crops only** — never val or test. Three sources:

1. **Positive crops** — every ground-truth fire/smoke box in the train split, expanded by 20% margin (matches inference-time `crop_margin`). Labels: `smoke=0`, `fire=1`.
2. **Mined hard negatives** — run YOLO on every train *negative* image (empty label file) at `mining_conf=0.10`. Every detection on a negative image is by definition a false positive — exactly the confounder distribution we want the probe to learn to reject. Label: `neg=2`.
3. **Random negative crops** — random ~200×200 patches from train negative images, capped at `max_random_negatives`. Adds breadth (generic forest, sky, structures) so the probe doesn't just learn "looks like a YOLO FP".

All crops are encoded with frozen CLIP ViT-B/32 once, then the probe trains on the cached 512-d features.

In [7]:
def extract_crop(image_pil, box_xyxy, margin=CONFIG["crop_margin"]):
    # Same expansion logic as improvement1_final.ipynb cell 18.
    W, H = image_pil.size
    x1, y1, x2, y2 = box_xyxy
    bw, bh = x2 - x1, y2 - y1
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    nw, nh = bw * (1 + margin), bh * (1 + margin)
    nx1 = max(0, int(cx - nw / 2)); ny1 = max(0, int(cy - nh / 2))
    nx2 = min(W, int(cx + nw / 2)); ny2 = min(H, int(cy + nh / 2))
    if nx2 <= nx1 or ny2 <= ny1:
        nx1, ny1, nx2, ny2 = int(x1), int(y1), int(x2), int(y2)
    return image_pil.crop((nx1, ny1, nx2, ny2))


@torch.no_grad()
def encode_crops_batched(crops_pil, batch_size=128):
    # Encode a list of PIL crops with frozen CLIP, return L2-normalized features (N, D).
    feats = []
    for i in range(0, len(crops_pil), batch_size):
        batch = torch.stack([clip_preprocess(c) for c in crops_pil[i:i+batch_size]]).to(device)
        f = clip_model.encode_image(batch)
        f = f / f.norm(dim=-1, keepdim=True)
        feats.append(f.float().cpu().numpy())
    return np.concatenate(feats, axis=0) if feats else np.zeros((0, CLIP_FEAT_DIM), dtype=np.float32)


def yolo_xywhn_to_xyxy(xc, yc, w, h, W, H):
    return ((xc - w/2) * W, (yc - h/2) * H, (xc + w/2) * W, (yc + h/2) * H)


print("Helpers defined: extract_crop, encode_crops_batched.")


Helpers defined: extract_crop, encode_crops_batched.


In [8]:
# Positive crops: walk train labels, extract every GT box.
print("Extracting positive train crops (this is the slowest part — ~3-6 min on T4)...")

pos_crops, pos_labels = [], []
train_lbl_files = [f for f in os.listdir(train_lbl) if f.endswith('.txt')]

for fname in tqdm(train_lbl_files, desc="train labels", unit="lbl"):
    lbl_path = os.path.join(train_lbl, fname)
    if os.path.getsize(lbl_path) == 0:
        continue  # negative — handled separately
    stem = os.path.splitext(fname)[0]
    img_path = None
    for ext in ('.jpg', '.jpeg', '.png'):
        cand = os.path.join(train_img, stem + ext)
        if os.path.exists(cand):
            img_path = cand; break
    if img_path is None: continue
    try:
        pil = Image.open(img_path).convert("RGB")
    except Exception:
        continue
    W, H = pil.size
    with open(lbl_path) as fh:
        for line in fh:
            parts = line.strip().split()
            if len(parts) < 5: continue
            try:
                cls = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:5])
            except ValueError:
                continue
            if cls not in (0, 1): continue
            box = yolo_xywhn_to_xyxy(xc, yc, bw, bh, W, H)
            crop = extract_crop(pil, box, margin=CONFIG["crop_margin"])
            if crop.size[0] < 4 or crop.size[1] < 4:
                continue
            pos_crops.append(crop)
            pos_labels.append(cls)   # 0=smoke, 1=fire

# Optional cap per class
if CONFIG["max_positives_per_cls"] is not None:
    rng = random.Random(SEED)
    by_cls = {0: [], 1: []}
    for c, lbl in zip(pos_crops, pos_labels):
        by_cls[lbl].append(c)
    capped_crops, capped_labels = [], []
    for cls, lst in by_cls.items():
        rng.shuffle(lst)
        capped = lst[:CONFIG["max_positives_per_cls"]]
        capped_crops.extend(capped)
        capped_labels.extend([cls] * len(capped))
    pos_crops, pos_labels = capped_crops, capped_labels

print(f"\nPositive crops collected: {len(pos_crops)}  "
      f"(smoke={sum(1 for l in pos_labels if l==0)}, fire={sum(1 for l in pos_labels if l==1)})")

print("Encoding positive crops with CLIP...")
pos_features = encode_crops_batched(pos_crops, batch_size=128)
print(f"Positive features shape: {pos_features.shape}")
del pos_crops  # free memory


Extracting positive train crops (this is the slowest part — ~3-6 min on T4)...


train labels: 100%|██████████| 14122/14122 [02:58<00:00, 79.32lbl/s] 



Positive crops collected: 17417  (smoke=7788, fire=9629)
Encoding positive crops with CLIP...
Positive features shape: (17417, 512)


In [9]:
# Mined hard negatives: YOLO predictions on train negative images.
# These are EXACTLY the confounder crops we need the probe to reject.
print("Mining hard negatives (YOLO FPs on train negatives)...")

neg_train_imgs = []
for fname in os.listdir(train_lbl):
    if not fname.endswith('.txt'): continue
    lbl_path = os.path.join(train_lbl, fname)
    if os.path.getsize(lbl_path) > 0: continue
    stem = os.path.splitext(fname)[0]
    for ext in ('.jpg', '.jpeg', '.png'):
        cand = os.path.join(train_img, stem + ext)
        if os.path.exists(cand):
            neg_train_imgs.append(cand); break

print(f"Train negative images: {len(neg_train_imgs)}")

mined_crops = []
mining_conf = CONFIG["mining_conf"]
for path in tqdm(neg_train_imgs, desc="mining", unit="img"):
    pred = yolo.predict(source=path, imgsz=CONFIG["imgsz"], conf=mining_conf,
                        iou=CONFIG["yolo_iou"], device=device, verbose=False)[0]
    if len(pred.boxes) == 0:
        continue
    try:
        pil = Image.open(path).convert("RGB")
    except Exception:
        continue
    for b in pred.boxes:
        x1, y1, x2, y2 = b.xyxy[0].cpu().numpy()
        crop = extract_crop(pil, (x1, y1, x2, y2), margin=CONFIG["crop_margin"])
        if crop.size[0] < 4 or crop.size[1] < 4:
            continue
        mined_crops.append(crop)

if CONFIG["max_mined_negatives"] is not None and len(mined_crops) > CONFIG["max_mined_negatives"]:
    rng = random.Random(SEED)
    rng.shuffle(mined_crops)
    mined_crops = mined_crops[:CONFIG["max_mined_negatives"]]

print(f"Mined hard-negative crops: {len(mined_crops)}")

print("Encoding mined negatives with CLIP...")
mined_features = encode_crops_batched(mined_crops, batch_size=128)
print(f"Mined features shape: {mined_features.shape}")
del mined_crops


Mining hard negatives (YOLO FPs on train negatives)...
Train negative images: 6458


mining: 100%|██████████| 6458/6458 [02:31<00:00, 42.55img/s]


Mined hard-negative crops: 282
Encoding mined negatives with CLIP...
Mined features shape: (282, 512)


In [10]:
# Random negatives: random crops from train negative images for breadth.
print("Sampling random negative crops...")

rng = random.Random(SEED)
target_n = CONFIG["max_random_negatives"]
random_crops = []

n_per_img = max(1, target_n // max(1, len(neg_train_imgs)) + 1)
neg_imgs_shuffled = neg_train_imgs[:]
rng.shuffle(neg_imgs_shuffled)

for path in tqdm(neg_imgs_shuffled, desc="random", unit="img"):
    if len(random_crops) >= target_n: break
    try:
        pil = Image.open(path).convert("RGB")
    except Exception:
        continue
    W, H = pil.size
    if W < 64 or H < 64: continue
    for _ in range(n_per_img):
        if len(random_crops) >= target_n: break
        cw = rng.randint(64, max(64, W // 2))
        ch = rng.randint(64, max(64, H // 2))
        x1 = rng.randint(0, W - cw); y1 = rng.randint(0, H - ch)
        crop = pil.crop((x1, y1, x1 + cw, y1 + ch))
        random_crops.append(crop)

print(f"Random negative crops: {len(random_crops)}")
print("Encoding random negatives with CLIP...")
random_features = encode_crops_batched(random_crops, batch_size=128)
print(f"Random features shape: {random_features.shape}")
del random_crops


Sampling random negative crops...


random:  77%|███████▋  | 5000/6458 [00:26<00:07, 187.06img/s]


Random negative crops: 5000
Encoding random negatives with CLIP...
Random features shape: (5000, 512)


In [11]:
# Assemble training arrays for the probe.
# Class encoding: 0=smoke, 1=fire, 2=neg.
neg_features_full = np.concatenate([mined_features, random_features], axis=0)

X = np.concatenate([pos_features, neg_features_full], axis=0).astype(np.float32)
y = np.array(pos_labels + [2] * neg_features_full.shape[0], dtype=np.int64)

# Shuffle
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(y))
X, y = X[perm], y[perm]

n_smoke = int((y == 0).sum())
n_fire  = int((y == 1).sum())
n_neg   = int((y == 2).sum())
print(f"Training set assembled:")
print(f"  smoke: {n_smoke}")
print(f"  fire : {n_fire}")
print(f"  neg  : {n_neg}  (mined={mined_features.shape[0]}, random={random_features.shape[0]})")
print(f"  total: {len(y)}, dim={X.shape[1]}")

# Save train arrays so retraining (e.g. for ablations) doesn't redo extraction.
np.savez_compressed(os.path.join(WORK_DIR, "probe_train_data.npz"),
                    X=X, y=y, mined_n=mined_features.shape[0],
                    random_n=random_features.shape[0])
print(f"Saved train data to {WORK_DIR}/probe_train_data.npz")


Training set assembled:
  smoke: 7788
  fire : 9629
  neg  : 5282  (mined=282, random=5000)
  total: 22699, dim=512
Saved train data to /kaggle/working/probe_train_data.npz


## 8. Train probes

Default probe is **logistic regression** (3-class, multinomial) with `class_weight='balanced'` to handle the smoke/fire/neg imbalance. Trains in seconds; small enough to never overfit a 512-d feature space.

The MLP probe is a 1-layer 64-unit MLP with dropout, trained with class-weighted cross-entropy. Used for the probe-type ablation.

In [12]:
# Internal 90/10 split for held-out validation accuracy.
# Stratified to keep class proportions.
def stratified_split(X, y, val_frac=0.1, seed=SEED):
    rng = np.random.default_rng(seed)
    idx_train, idx_val = [], []
    for c in np.unique(y):
        idx_c = np.where(y == c)[0]
        rng.shuffle(idx_c)
        n_val = max(1, int(len(idx_c) * val_frac))
        idx_val.extend(idx_c[:n_val].tolist())
        idx_train.extend(idx_c[n_val:].tolist())
    rng.shuffle(idx_train); rng.shuffle(idx_val)
    return np.array(idx_train), np.array(idx_val)

idx_tr, idx_va = stratified_split(X, y, val_frac=0.1, seed=SEED)
X_tr, y_tr = X[idx_tr], y[idx_tr]
X_va, y_va = X[idx_va], y[idx_va]
print(f"Probe internal split: train={len(idx_tr)}, val={len(idx_va)}")

# --- Logistic regression probe ---
print("\nFitting logistic regression probe...")
t0 = time.time()
probe_lr = LogisticRegression(
    C=CONFIG["lr_C"],
    max_iter=CONFIG["lr_max_iter"],
    multi_class="multinomial",
    class_weight="balanced",
    solver="lbfgs",
    random_state=SEED,
)
probe_lr.fit(X_tr, y_tr)
print(f"LR fit in {time.time()-t0:.1f}s")

y_pred = probe_lr.predict(X_va)
print("\nLogistic-regression probe — held-out 10% accuracy:")
print(classification_report(y_va, y_pred, target_names=PROBE_CLASS_NAMES, digits=4))

with open(CONFIG["probe_lr_path"], "wb") as f:
    pickle.dump(probe_lr, f)
print(f"Saved LR probe to {CONFIG['probe_lr_path']}")


Probe internal split: train=20431, val=2268

Fitting logistic regression probe...
LR fit in 4.9s

Logistic-regression probe — held-out 10% accuracy:
              precision    recall  f1-score   support

       smoke     0.9179    0.8907    0.9041       778
        fire     0.9450    0.9470    0.9460       962
         neg     0.9144    0.9508    0.9322       528

    accuracy                         0.9286      2268
   macro avg     0.9258    0.9295    0.9274      2268
weighted avg     0.9286    0.9286    0.9284      2268

Saved LR probe to /kaggle/working/probe_lr.pkl


In [13]:
# --- MLP probe (ablation) ---
class MLPProbe(nn.Module):
    def __init__(self, d_in, d_hidden, n_classes, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, n_classes),
        )
    def forward(self, x):
        return self.net(x)

class_counts = np.bincount(y_tr, minlength=3).astype(np.float32)
class_w = (len(y_tr) / (3 * class_counts))
class_w_t = torch.tensor(class_w, device=device, dtype=torch.float32)
print(f"Class weights for MLP CE: {class_w}")

mlp = MLPProbe(CLIP_FEAT_DIM, CONFIG["mlp_hidden"], 3, CONFIG["mlp_dropout"]).to(device)
opt = torch.optim.Adam(mlp.parameters(), lr=CONFIG["mlp_lr"], weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss(weight=class_w_t)

X_tr_t = torch.from_numpy(X_tr).to(device)
y_tr_t = torch.from_numpy(y_tr).to(device)
X_va_t = torch.from_numpy(X_va).to(device)
y_va_t = torch.from_numpy(y_va).to(device)

bs = CONFIG["mlp_batch_size"]
print("\nTraining MLP probe...")
t0 = time.time()
best_va_acc = 0.0
for ep in range(CONFIG["mlp_epochs"]):
    mlp.train()
    perm = torch.randperm(len(X_tr_t), device=device)
    total_loss = 0.0
    for i in range(0, len(perm), bs):
        sl = perm[i:i+bs]
        opt.zero_grad()
        logits = mlp(X_tr_t[sl])
        loss = loss_fn(logits, y_tr_t[sl])
        loss.backward(); opt.step()
        total_loss += loss.item() * len(sl)
    mlp.eval()
    with torch.no_grad():
        va_pred = mlp(X_va_t).argmax(-1)
        va_acc = (va_pred == y_va_t).float().mean().item()
    if va_acc > best_va_acc:
        best_va_acc = va_acc
        torch.save(mlp.state_dict(), CONFIG["probe_mlp_path"])
    if (ep + 1) % 5 == 0 or ep == 0:
        print(f"  epoch {ep+1:3d}/{CONFIG['mlp_epochs']}  "
              f"loss={total_loss/len(X_tr_t):.4f}  val_acc={va_acc:.4f}")
print(f"MLP fit in {time.time()-t0:.1f}s, best_val_acc={best_va_acc:.4f}")

# Load best
mlp.load_state_dict(torch.load(CONFIG["probe_mlp_path"], map_location=device))
mlp.eval()
print(f"Saved best MLP probe to {CONFIG['probe_mlp_path']}")

with torch.no_grad():
    va_pred = mlp(X_va_t).argmax(-1).cpu().numpy()
print("\nMLP probe — held-out 10% accuracy:")
print(classification_report(y_va, va_pred, target_names=PROBE_CLASS_NAMES, digits=4))


Class weights for MLP CE: [    0.97152     0.78578      1.4325]

Training MLP probe...
  epoch   1/40  loss=1.0564  val_acc=0.7875
  epoch   5/40  loss=0.4998  val_acc=0.8554
  epoch  10/40  loss=0.3048  val_acc=0.9034
  epoch  15/40  loss=0.2573  val_acc=0.9211
  epoch  20/40  loss=0.2376  val_acc=0.9255
  epoch  25/40  loss=0.2224  val_acc=0.9299
  epoch  30/40  loss=0.2131  val_acc=0.9321
  epoch  35/40  loss=0.2047  val_acc=0.9374
  epoch  40/40  loss=0.1979  val_acc=0.9405
MLP fit in 1.7s, best_val_acc=0.9405
Saved best MLP probe to /kaggle/working/probe_mlp.pt

MLP probe — held-out 10% accuracy:
              precision    recall  f1-score   support

       smoke     0.9249    0.9177    0.9213       778
        fire     0.9582    0.9543    0.9563       962
         neg     0.9312    0.9489    0.9400       528

    accuracy                         0.9405      2268
   macro avg     0.9381    0.9403    0.9392      2268
weighted avg     0.9405    0.9405    0.9405      2268



## 9. Cascade integration

Three pieces. Each operates on CLIP features of the cropped detections:

- `clip_verify_text` — I1's zero-shot text-prompt argmax (copied verbatim, modulo light refactor to share encoding).
- `clip_verify_probe_lr` — argmax of the trained logistic-regression probe.
- `clip_verify_probe_mlp` — argmax of the trained MLP probe.

All three return the same dict shape (one per crop), so the rest of the cascade is unchanged.

In [14]:
@torch.no_grad()
def encode_detection_crops(image_pil, boxes_xyxy, crop_margin):
    # Crop, preprocess, encode with CLIP. Returns L2-normalized features (N, D).
    if len(boxes_xyxy) == 0:
        return np.zeros((0, CLIP_FEAT_DIM), dtype=np.float32)
    crops = [extract_crop(image_pil, b, margin=crop_margin) for b in boxes_xyxy]
    crops_t = torch.stack([clip_preprocess(c) for c in crops]).to(device)
    img_feats = clip_model.encode_image(crops_t)
    img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
    return img_feats  # keep on GPU; verifiers can move to CPU as needed


def clip_verify_text(image_pil, boxes_xyxy, crop_margin, *, text_feats=text_feats):
    # I1's zero-shot verifier (re-implemented over the shared encoder).
    feats = encode_detection_crops(image_pil, boxes_xyxy, crop_margin)
    if feats.shape[0] == 0: return []
    sims = (feats @ text_feats.T).cpu().numpy()
    out = []
    for row in sims:
        top = int(np.argmax(row))
        out.append({
            "top_idx":     top,
            "top_bucket":  bucket_of_text(top),
            "top_sim":     float(row[top]),
            "fire_score":  float(row[FIRE_IDX].max()),
            "smoke_score": float(row[SMOKE_IDX].max()),
            "neg_score":   float(row[NEG_IDX].max()),
        })
    return out


def make_clip_verify_probe_lr(probe):
    def _verify(image_pil, boxes_xyxy, crop_margin):
        feats_t = encode_detection_crops(image_pil, boxes_xyxy, crop_margin)
        if feats_t.shape[0] == 0: return []
        feats = feats_t.float().cpu().numpy()
        proba = probe.predict_proba(feats)   # (N, 3) over [smoke, fire, neg]
        out = []
        for row in proba:
            top = int(np.argmax(row))
            out.append({
                "top_idx":    top,
                "top_bucket": PROBE_CLASS_NAMES[top],
                "top_score":  float(row[top]),
                "smoke_score":float(row[0]),
                "fire_score": float(row[1]),
                "neg_score":  float(row[2]),
            })
        return out
    return _verify


def make_clip_verify_probe_mlp(mlp_module):
    @torch.no_grad()
    def _verify(image_pil, boxes_xyxy, crop_margin):
        feats_t = encode_detection_crops(image_pil, boxes_xyxy, crop_margin)
        if feats_t.shape[0] == 0: return []
        logits = mlp_module(feats_t.float())
        proba = F.softmax(logits, dim=-1).cpu().numpy()
        out = []
        for row in proba:
            top = int(np.argmax(row))
            out.append({
                "top_idx":    top,
                "top_bucket": PROBE_CLASS_NAMES[top],
                "top_score":  float(row[top]),
                "smoke_score":float(row[0]),
                "fire_score": float(row[1]),
                "neg_score":  float(row[2]),
            })
        return out
    return _verify


def verify_detection(yolo_class_id, clip_result):
    # Same logic as I1: keep iff CLIP/probe argmax bucket matches YOLO class.
    expected = CLASS_NAMES[yolo_class_id]
    return clip_result["top_bucket"] == expected


# Build the verifiers
verify_text     = clip_verify_text
verify_probe_lr = make_clip_verify_probe_lr(probe_lr)
verify_probe_mlp= make_clip_verify_probe_mlp(mlp)
print("Verifiers ready: verify_text, verify_probe_lr, verify_probe_mlp")


Verifiers ready: verify_text, verify_probe_lr, verify_probe_mlp


In [15]:
def yolo_clip_pipeline_v2(
    image_path, verify_fn,
    conf=CONFIG["yolo_conf"], iou=CONFIG["yolo_iou"], imgsz=CONFIG["imgsz"],
    crop_margin=CONFIG["crop_margin"], tau_high=CONFIG["tau_high"],
):
    # Generalized cascaded pipeline — identical scaffold to I1's yolo_clip_pipeline,
    # parameterized over the verifier so we can swap text-prompt vs probe.
    #   yolo_conf >= tau_high -> auto-keep (skip verifier)
    #   yolo_conf <  tau_high -> verifier; keep iff verifier bucket == YOLO class
    t0 = time.perf_counter()
    pred = yolo.predict(source=image_path, imgsz=imgsz, conf=conf, iou=iou,
                        device=device, verbose=False)[0]
    t1 = time.perf_counter()

    yolo_boxes = []
    for b in pred.boxes:
        x1, y1, x2, y2 = b.xyxy[0].cpu().numpy()
        yolo_boxes.append({
            "x1": float(x1), "y1": float(y1), "x2": float(x2), "y2": float(y2),
            "cls": int(b.cls[0]), "conf": float(b.conf[0]),
        })
    if not yolo_boxes:
        return {"yolo_boxes": [], "kept_indices": [], "gated_indices": [],
                "verified_indices": [], "clip_results": [],
                "timing": {"yolo_ms": (t1-t0)*1000, "clip_ms": 0.0,
                           "total_ms": (t1-t0)*1000}}

    gated_idx    = [i for i, b in enumerate(yolo_boxes) if b["conf"] >= tau_high]
    verified_idx = [i for i, b in enumerate(yolo_boxes) if b["conf"] <  tau_high]
    clip_results = [None] * len(yolo_boxes)

    t2 = time.perf_counter()
    if verified_idx:
        pil = Image.open(image_path).convert("RGB")
        boxes_xy = [(yolo_boxes[i]["x1"], yolo_boxes[i]["y1"],
                     yolo_boxes[i]["x2"], yolo_boxes[i]["y2"]) for i in verified_idx]
        results = verify_fn(pil, boxes_xy, crop_margin)
        for j, i in enumerate(verified_idx):
            clip_results[i] = results[j]
    t3 = time.perf_counter()

    kept = []
    for i, yb in enumerate(yolo_boxes):
        if clip_results[i] is None:
            kept.append(i)
        else:
            if verify_detection(yb["cls"], clip_results[i]):
                kept.append(i)

    return {"yolo_boxes": yolo_boxes, "kept_indices": kept,
            "gated_indices": gated_idx, "verified_indices": verified_idx,
            "clip_results": clip_results,
            "timing": {"yolo_ms": (t1-t0)*1000, "clip_ms": (t3-t2)*1000,
                       "total_ms": (t3-t0)*1000}}

print("Cascaded pipeline v2 ready.")


Cascaded pipeline v2 ready.


## 10. Evaluation helpers (copied from I1 cell 30-31 verbatim)

These are the AP / matching / FPR helpers from `improvement1_final.ipynb`. Keeping them byte-for-byte means I2 vs I1 numbers from this notebook are produced by the **same harness** — the comparison is apples-to-apples even where Ultralytics' native `val()` would disagree.

In [16]:
def collect_hard_negatives(test_img_dir, test_lbl_dir):
    neg_imgs = []
    for fname in os.listdir(test_lbl_dir):
        if not fname.endswith('.txt'): continue
        lbl_path = os.path.join(test_lbl_dir, fname)
        if os.path.getsize(lbl_path) > 0: continue
        stem = os.path.splitext(fname)[0]
        for ext in ('.jpg', '.jpeg', '.png'):
            cand = os.path.join(test_img_dir, stem + ext)
            if os.path.exists(cand):
                neg_imgs.append(cand); break
    return neg_imgs


def load_gt_boxes(lbl_path, img_w, img_h):
    gts = []
    if not os.path.exists(lbl_path) or os.path.getsize(lbl_path) == 0:
        return gts
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5: continue
            try:
                cls = int(parts[0])
                xc, yc, w, h = map(float, parts[1:5])
            except ValueError:
                continue
            x1 = (xc - w/2) * img_w; y1 = (yc - h/2) * img_h
            x2 = (xc + w/2) * img_w; y2 = (yc + h/2) * img_h
            gts.append((cls, x1, y1, x2, y2))
    return gts


def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    union = (max(0, ax2-ax1) * max(0, ay2-ay1) +
             max(0, bx2-bx1) * max(0, by2-by1) - inter)
    return inter / union if union > 0 else 0.0


def match_predictions(pred_boxes, gt_boxes, iou_thresh=0.5):
    matched_gt = set(); tps = []
    for cls, x1, y1, x2, y2, _ in pred_boxes:
        best_iou, best_j = 0.0, -1
        for j, (gcls, gx1, gy1, gx2, gy2) in enumerate(gt_boxes):
            if j in matched_gt: continue
            if gcls != cls: continue
            iou = box_iou((x1, y1, x2, y2), (gx1, gy1, gx2, gy2))
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_iou >= iou_thresh and best_j >= 0:
            tps.append(True); matched_gt.add(best_j)
        else:
            tps.append(False)
    return tps


def compute_ap(preds, n_total_gt):
    if n_total_gt == 0 or not preds:
        return 0.0, 0.0, 0.0
    preds_sorted = sorted(preds, key=lambda x: -x[0])
    tp_cum = fp_cum = 0
    precisions, recalls = [], []
    for _, tp in preds_sorted:
        if tp: tp_cum += 1
        else:  fp_cum += 1
        precisions.append(tp_cum / (tp_cum + fp_cum))
        recalls.append(tp_cum / n_total_gt)
    ap = 0.0
    for t in np.linspace(0, 1, 11):
        p_at = max([p for p, r in zip(precisions, recalls) if r >= t], default=0.0)
        ap += p_at / 11
    return ap, (precisions[-1] if precisions else 0.0), (recalls[-1] if recalls else 0.0)


In [17]:
def evaluate_fpr_on_negatives(
    neg_imgs, verify_fn, *,
    crop_margin=CONFIG["crop_margin"], conf=CONFIG["yolo_conf"],
    iou=CONFIG["yolo_iou"], tau_high=CONFIG["tau_high"], desc="",
):
    # Image-level + box-level FPR on hard-negative subset, identical to I1 cell 26 logic.
    yolo_fp_imgs = yolo_fp_boxes = 0
    fp_imgs_after = fp_boxes_after = boxes_filtered = 0
    n_gated = n_verified = 0
    latencies = []
    for path in tqdm(neg_imgs, desc=desc, unit="img"):
        out = yolo_clip_pipeline_v2(path, verify_fn,
                                    conf=conf, iou=iou, imgsz=CONFIG["imgsz"],
                                    crop_margin=crop_margin, tau_high=tau_high)
        latencies.append(out["timing"]["total_ms"])
        n_gated    += len(out["gated_indices"])
        n_verified += len(out["verified_indices"])
        n_yolo = len(out["yolo_boxes"])
        n_kept = len(out["kept_indices"])
        if n_yolo > 0: yolo_fp_imgs += 1; yolo_fp_boxes += n_yolo
        if n_kept > 0: fp_imgs_after += 1; fp_boxes_after += n_kept
        boxes_filtered += (n_yolo - n_kept)
    n = len(neg_imgs)
    return {
        "n_negatives":   n,
        "yolo_fp_imgs":  yolo_fp_imgs,
        "yolo_fp_boxes": yolo_fp_boxes,
        "yolo_fpr":      yolo_fp_imgs / n if n else 0.0,
        "fp_imgs_after": fp_imgs_after,
        "fp_boxes_after":fp_boxes_after,
        "fpr_after":     fp_imgs_after / n if n else 0.0,
        "boxes_filtered":boxes_filtered,
        "n_gated":       n_gated,
        "n_verified":    n_verified,
        "mean_latency_ms": float(np.mean(latencies)) if latencies else 0.0,
    }


def full_test_eval(test_img_dir, test_lbl_dir, verify_fn, *,
                   crop_margin=CONFIG["crop_margin"], conf=CONFIG["yolo_conf"],
                   iou=CONFIG["yolo_iou"], tau_high=CONFIG["tau_high"],
                   max_images=None, desc=""):
    # Full-test mAP/precision/recall under our custom 11-pt-AP harness (identical to I1 cell 31).
    img_files = sorted([f for f in os.listdir(test_img_dir)
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if max_images is not None:
        img_files = img_files[:max_images]

    yolo_preds = {0: [], 1: []}
    cascade_preds = {0: [], 1: []}
    n_gt = {0: 0, 1: 0}
    latencies = []

    for fname in tqdm(img_files, desc=f"full eval {desc}", unit="img"):
        path = os.path.join(test_img_dir, fname)
        stem = os.path.splitext(fname)[0]
        lbl_path = os.path.join(test_lbl_dir, stem + '.txt')
        try:
            pil = Image.open(path); img_w, img_h = pil.size; pil.close()
        except Exception:
            continue
        gt_boxes = load_gt_boxes(lbl_path, img_w, img_h)
        for (c, *_) in gt_boxes:
            if c in n_gt: n_gt[c] += 1

        out = yolo_clip_pipeline_v2(path, verify_fn,
                                    conf=conf, iou=iou, imgsz=CONFIG["imgsz"],
                                    crop_margin=crop_margin, tau_high=tau_high)
        latencies.append(out["timing"]["total_ms"])

        yolo_only = [(b["cls"], b["x1"], b["y1"], b["x2"], b["y2"], b["conf"])
                     for b in out["yolo_boxes"]]
        yolo_only.sort(key=lambda x: -x[5])
        cascade_kept = [(out["yolo_boxes"][i]["cls"],
                         out["yolo_boxes"][i]["x1"], out["yolo_boxes"][i]["y1"],
                         out["yolo_boxes"][i]["x2"], out["yolo_boxes"][i]["y2"],
                         out["yolo_boxes"][i]["conf"])
                        for i in out["kept_indices"]]
        cascade_kept.sort(key=lambda x: -x[5])

        for cls in [0, 1]:
            yolo_cls = [p for p in yolo_only if p[0] == cls]
            cas_cls  = [p for p in cascade_kept if p[0] == cls]
            gt_cls   = [g for g in gt_boxes if g[0] == cls]
            if yolo_cls:
                tps = match_predictions(yolo_cls, gt_cls)
                for p, tp in zip(yolo_cls, tps):
                    yolo_preds[cls].append((p[5], tp))
            if cas_cls:
                tps = match_predictions(cas_cls, gt_cls)
                for p, tp in zip(cas_cls, tps):
                    cascade_preds[cls].append((p[5], tp))

    out = {"yolo": {}, "cascade": {}}
    for name, preds in [("yolo", yolo_preds), ("cascade", cascade_preds)]:
        aps, ps, rs = [], [], []
        for cls in [0, 1]:
            ap, p, r = compute_ap(preds[cls], n_gt[cls])
            out[name][f"ap50_{CLASS_NAMES[cls]}"]      = ap
            out[name][f"precision_{CLASS_NAMES[cls]}"] = p
            out[name][f"recall_{CLASS_NAMES[cls]}"]    = r
            aps.append(ap); ps.append(p); rs.append(r)
        out[name]["mAP50"]          = float(np.mean(aps))
        out[name]["mean_precision"] = float(np.mean(ps))
        out[name]["mean_recall"]    = float(np.mean(rs))
    out["n_gt"]        = n_gt
    out["n_images"]    = len(img_files)
    out["tau_high"]    = tau_high
    out["mean_lat_ms"] = float(np.mean(latencies))
    out["p95_lat_ms"]  = float(np.percentile(latencies, 95))
    return out

print("Eval helpers ready: evaluate_fpr_on_negatives, full_test_eval.")


Eval helpers ready: evaluate_fpr_on_negatives, full_test_eval.


## 11. Hard-negative FPR — the headline metric

Compute FPR on the 2,005 D-Fire test images with empty labels for **three pipelines**, on the same harness:

1. YOLO-only (the `yolo` field of `evaluate_fpr_on_negatives` already gives us this for free in every run).
2. **Improvement 1 zero-shot CLIP** (re-baselined inside this notebook).
3. **Improvement 2 logistic-regression probe** (the new contribution).

In [18]:
neg_test_imgs = collect_hard_negatives(test_img, test_lbl)
print(f"Hard-negative test images: {len(neg_test_imgs)}")

print("\n=== Hard-neg FPR: I1 zero-shot text prompts ===")
fpr_text = evaluate_fpr_on_negatives(neg_test_imgs, verify_text, desc="I1 text")
print(json.dumps(fpr_text, indent=2))

print("\n=== Hard-neg FPR: I2 logistic-regression probe ===")
fpr_probe_lr = evaluate_fpr_on_negatives(neg_test_imgs, verify_probe_lr, desc="I2 LR")
print(json.dumps(fpr_probe_lr, indent=2))


Hard-negative test images: 2005

=== Hard-neg FPR: I1 zero-shot text prompts ===


I1 text: 100%|██████████| 2005/2005 [00:26<00:00, 74.91img/s]


{
  "n_negatives": 2005,
  "yolo_fp_imgs": 49,
  "yolo_fp_boxes": 55,
  "yolo_fpr": 0.024438902743142144,
  "fp_imgs_after": 11,
  "fp_boxes_after": 11,
  "fpr_after": 0.005486284289276808,
  "boxes_filtered": 44,
  "n_gated": 2,
  "n_verified": 53,
  "mean_latency_ms": 13.176612679801469
}

=== Hard-neg FPR: I2 logistic-regression probe ===


I2 LR: 100%|██████████| 2005/2005 [00:25<00:00, 78.45img/s]

{
  "n_negatives": 2005,
  "yolo_fp_imgs": 49,
  "yolo_fp_boxes": 55,
  "yolo_fpr": 0.024438902743142144,
  "fp_imgs_after": 25,
  "fp_boxes_after": 27,
  "fpr_after": 0.012468827930174564,
  "boxes_filtered": 28,
  "n_gated": 2,
  "n_verified": 53,
  "mean_latency_ms": 12.606559554113973
}


## 12. Full test set evaluation (mAP / precision / recall)

The eval that took ~8–15 min per pass in I1. Run twice: once with the I1 zero-shot verifier, once with the I2 logistic-regression probe. The harness is identical (custom 11-pt AP, IoU=0.5 greedy class-conditional matcher) — so the I1 vs I2 delta here is a fair comparison.

In [19]:
print("=== Full test eval: I1 zero-shot ===")
full_text = full_test_eval(test_img, test_lbl, verify_text,
                           max_images=CONFIG["max_eval_images"], desc="I1")
print(json.dumps({k: v for k, v in full_text.items() if k != "n_gt"}, indent=2))

print("\n=== Full test eval: I2 LR probe ===")
full_probe_lr = full_test_eval(test_img, test_lbl, verify_probe_lr,
                               max_images=CONFIG["max_eval_images"], desc="I2 LR")
print(json.dumps({k: v for k, v in full_probe_lr.items() if k != "n_gt"}, indent=2))


=== Full test eval: I1 zero-shot ===


full eval I1: 100%|██████████| 4306/4306 [01:41<00:00, 42.25img/s]


{
  "yolo": {
    "ap50_smoke": 0.6948293172257434,
    "precision_smoke": 0.8074041427941825,
    "recall_smoke": 0.7913606911447084,
    "ap50_fire": 0.5791710207045309,
    "precision_fire": 0.6888086642599278,
    "recall_fire": 0.6629603891591382,
    "mAP50": 0.6370001689651371,
    "mean_precision": 0.7481064035270552,
    "mean_recall": 0.7271605401519233
  },
  "cascade": {
    "ap50_smoke": 0.6114310287516386,
    "precision_smoke": 0.8760951188986232,
    "recall_smoke": 0.6047516198704104,
    "ap50_fire": 0.5003554856966653,
    "precision_fire": 0.7421913733267228,
    "recall_fire": 0.5201528839471855,
    "mAP50": 0.555893257224152,
    "mean_precision": 0.8091432461126731,
    "mean_recall": 0.562452251908798
  },
  "n_images": 4306,
  "tau_high": 0.7,
  "mean_lat_ms": 20.81174758755207,
  "p95_lat_ms": 42.671980500017526
}

=== Full test eval: I2 LR probe ===


full eval I2 LR: 100%|██████████| 4306/4306 [01:40<00:00, 42.72img/s] 

{
  "yolo": {
    "ap50_smoke": 0.6948293172257434,
    "precision_smoke": 0.8074041427941825,
    "recall_smoke": 0.7913606911447084,
    "ap50_fire": 0.5791710207045309,
    "precision_fire": 0.6888086642599278,
    "recall_fire": 0.6629603891591382,
    "mAP50": 0.6370001689651371,
    "mean_precision": 0.7481064035270552,
    "mean_recall": 0.7271605401519233
  },
  "cascade": {
    "ap50_smoke": 0.6951671931050467,
    "precision_smoke": 0.8362068965517241,
    "recall_smoke": 0.7542116630669546,
    "ap50_fire": 0.5778049360701977,
    "precision_fire": 0.6936736958934517,
    "recall_fire": 0.6514940931202223,
    "mAP50": 0.6364860645876222,
    "mean_precision": 0.764940296222588,
    "mean_recall": 0.7028528780935885
  },
  "n_images": 4306,
  "tau_high": 0.7,
  "mean_lat_ms": 21.197303641200577,
  "p95_lat_ms": 43.71813824991477
}


## 13. End-to-end latency benchmark

100 random test images, single-image inference (matches I1 cell 46). Reports YOLO-only, full-cascade mean and p95 in ms. The cascade adds CLIP encoding + linear-probe `predict_proba` on top of YOLO; the probe call is microseconds, so total latency should be very close to I1's (cascade is the cost driver, not the verifier).

In [20]:
all_test_imgs = [os.path.join(test_img, f) for f in os.listdir(test_img)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
bench_imgs = random.sample(all_test_imgs, k=min(100, len(all_test_imgs)))

# Warm up
for p in bench_imgs[:5]:
    _ = yolo_clip_pipeline_v2(p, verify_probe_lr)

yolo_lats, clip_lats, total_lats = [], [], []
for p in tqdm(bench_imgs, desc="latency", unit="img"):
    out = yolo_clip_pipeline_v2(p, verify_probe_lr)
    yolo_lats.append(out["timing"]["yolo_ms"])
    clip_lats.append(out["timing"]["clip_ms"])
    total_lats.append(out["timing"]["total_ms"])

latency_bench = {
    "yolo_mean_ms":   float(np.mean(yolo_lats)),
    "clip_mean_ms":   float(np.mean(clip_lats)),
    "total_mean_ms":  float(np.mean(total_lats)),
    "total_p95_ms":   float(np.percentile(total_lats, 95)),
    "n_images":       len(bench_imgs),
}
print(json.dumps(latency_bench, indent=2))
print(f"\n100ms budget: {'OK' if latency_bench['total_p95_ms'] < 100 else 'OVER'}")


latency: 100%|██████████| 100/100 [00:02<00:00, 47.62img/s]

{
  "yolo_mean_ms": 13.3922021699982,
  "clip_mean_ms": 7.261906740000086,
  "total_mean_ms": 20.808341209999526,
  "total_p95_ms": 43.70292620003511,
  "n_images": 100
}

100ms budget: OK


## 14. Ablations

Four ablations, each on the hard-negative subset only (cheap to run, ~1-2 min each):

1. **Probe type:** logistic regression vs MLP.
2. **τ_high sweep:** same grid as I1 — `[0.0, 0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]`.
3. **Crop-margin sweep:** same grid as I1 — `[0.00, 0.20, 0.40, 0.60]`.
4. **Negative-mining strategy:** mined-only (drop random negatives) vs mined+random (default).

In [21]:
# --- Ablation 1: probe type ---
print("=== Ablation 1: probe type (LR vs MLP) ===")
abl_probe_type = {
    "logistic_regression": evaluate_fpr_on_negatives(
        neg_test_imgs, verify_probe_lr, desc="LR"),
    "mlp": evaluate_fpr_on_negatives(
        neg_test_imgs, verify_probe_mlp, desc="MLP"),
}
for name, r in abl_probe_type.items():
    print(f"  {name:20s}  fpr={r['fpr_after']*100:.3f}%  "
          f"filtered={r['boxes_filtered']}  lat={r['mean_latency_ms']:.1f}ms")


=== Ablation 1: probe type (LR vs MLP) ===


MLP: 100%|██████████| 2005/2005 [00:27<00:00, 71.93img/s]

  logistic_regression   fpr=1.247%  filtered=28  lat=13.5ms
  mlp                   fpr=1.247%  filtered=28  lat=13.7ms


In [22]:
# --- Ablation 2: tau_high sweep (LR probe) ---
print("\n=== Ablation 2: tau_high sweep (LR probe) ===")
abl_tau = {}
for tau in CONFIG["tau_grid"]:
    print(f"\n--- tau_high = {tau:.2f} ---")
    r = evaluate_fpr_on_negatives(neg_test_imgs, verify_probe_lr,
                                  tau_high=tau, desc=f"tau={tau:.2f}")
    abl_tau[f"{tau:.2f}"] = r
    print(f"  fpr={r['fpr_after']*100:.3f}%  filtered={r['boxes_filtered']}  "
          f"verified={r['n_verified']}  lat={r['mean_latency_ms']:.1f}ms")



=== Ablation 2: tau_high sweep (LR probe) ===

--- tau_high = 0.00 ---


tau=0.00: 100%|██████████| 2005/2005 [00:26<00:00, 75.30img/s]


  fpr=2.444%  filtered=0  verified=0  lat=13.1ms

--- tau_high = 0.30 ---


tau=0.30: 100%|██████████| 2005/2005 [00:28<00:00, 71.32img/s]


  fpr=2.244%  filtered=5  verified=12  lat=13.8ms

--- tau_high = 0.50 ---


...

  fpr=1.197%  filtered=29  verified=54  lat=13.4ms

--- tau_high = 0.90 ---


tau=0.90: 100%|██████████| 2005/2005 [00:28<00:00, 71.05img/s]


  fpr=1.147%  filtered=30  verified=55  lat=13.9ms

--- tau_high = 1.00 ---


tau=1.00: 100%|██████████| 2005/2005 [00:27<00:00, 73.91img/s]

  fpr=1.147%  filtered=30  verified=55  lat=13.4ms


In [23]:
# --- Ablation 3: crop_margin sweep (LR probe) ---
print("\n=== Ablation 3: crop_margin sweep (LR probe) ===")
abl_margin = {}
for m in CONFIG["margin_grid"]:
    print(f"\n--- crop_margin = {m:.2f} ---")
    r = evaluate_fpr_on_negatives(neg_test_imgs, verify_probe_lr,
                                  crop_margin=m, desc=f"m={m:.2f}")
    abl_margin[f"{m:.2f}"] = r
    print(f"  fpr={r['fpr_after']*100:.3f}%  filtered={r['boxes_filtered']}  "
          f"lat={r['mean_latency_ms']:.1f}ms")



=== Ablation 3: crop_margin sweep (LR probe) ===

--- crop_margin = 0.00 ---


m=0.00: 100%|██████████| 2005/2005 [00:27<00:00, 73.50img/s]


  fpr=1.446%  filtered=24  lat=13.4ms

--- crop_margin = 0.20 ---


m=0.20: 100%|██████████| 2005/2005 [00:26<00:00, 76.22img/s]


  fpr=1.247%  filtered=28  lat=13.0ms

--- crop_margin = 0.40 ---


m=0.40: 100%|██████████| 2005/2005 [00:28<00:00, 70.95img/s]


  fpr=1.297%  filtered=27  lat=13.9ms

--- crop_margin = 0.60 ---


m=0.60: 100%|██████████| 2005/2005 [00:27<00:00, 73.57img/s]

  fpr=1.197%  filtered=29  lat=13.4ms


In [24]:
# --- Ablation 4: negative-mining strategy ---
# Re-train an LR probe with mined-only negatives (no random crops) and compare.
print("\n=== Ablation 4: mined-only vs mined+random ===")

mined_only_idx = np.concatenate([
    np.where(y == 0)[0],
    np.where(y == 1)[0],
    np.where(y == 2)[0][:mined_features.shape[0]],   # only the mined portion
])
X_mo, y_mo = X[mined_only_idx], y[mined_only_idx]
print(f"  mined-only train set: {len(y_mo)}  "
      f"(smoke={int((y_mo==0).sum())}, fire={int((y_mo==1).sum())}, neg={int((y_mo==2).sum())})")

probe_lr_mined = LogisticRegression(
    C=CONFIG["lr_C"], max_iter=CONFIG["lr_max_iter"],
    multi_class="multinomial", class_weight="balanced",
    solver="lbfgs", random_state=SEED,
)
probe_lr_mined.fit(X_mo, y_mo)
verify_probe_lr_mined = make_clip_verify_probe_lr(probe_lr_mined)

abl_neg_mining = {
    "mined_plus_random": evaluate_fpr_on_negatives(
        neg_test_imgs, verify_probe_lr, desc="mined+random"),
    "mined_only": evaluate_fpr_on_negatives(
        neg_test_imgs, verify_probe_lr_mined, desc="mined only"),
}
for name, r in abl_neg_mining.items():
    print(f"  {name:20s}  fpr={r['fpr_after']*100:.3f}%  "
          f"filtered={r['boxes_filtered']}  lat={r['mean_latency_ms']:.1f}ms")



=== Ablation 4: mined-only vs mined+random ===
  mined-only train set: 17699  (smoke=7788, fire=9629, neg=282)


mined only: 100%|██████████| 2005/2005 [00:27<00:00, 73.69img/s]

  mined_plus_random     fpr=1.247%  filtered=28  lat=12.9ms
  mined_only            fpr=1.397%  filtered=25  lat=13.4ms


## 15. Final comparison: Baseline vs Improvement 1 vs Improvement 2

The table below is produced by **the same harness** for the cascade rows (I1 zero-shot and I2 probe). The YOLO-only row uses Ultralytics' native eval (the canonical baseline reference, mAP50 ≈ 0.7614). The custom-harness YOLO-only number is also reported alongside as the apples-to-apples reference for the cascade comparisons.

In [25]:
# Headline numbers
def _pct(x): return f"{x*100:.2f}%"

rows = []
rows.append(("Baseline YOLO (yolo.val)", yolo_native["mAP50"], yolo_native["mean_precision"],
             yolo_native["mean_recall"], 2.44/100, "16.5 ms (T4)"))
# Custom-harness YOLO-only (from full_text["yolo"]) — same images and harness as cascade rows
rows.append(("YOLO-only (custom harness)", full_text["yolo"]["mAP50"], full_text["yolo"]["mean_precision"],
             full_text["yolo"]["mean_recall"], fpr_text["yolo_fpr"],
             f"{fpr_text['mean_latency_ms']:.1f} ms"))
# I1 zero-shot
rows.append(("I1 YOLO+CLIP (zero-shot)", full_text["cascade"]["mAP50"],
             full_text["cascade"]["mean_precision"], full_text["cascade"]["mean_recall"],
             fpr_text["fpr_after"],
             f"{full_text['mean_lat_ms']:.1f} ms (p95 {full_text['p95_lat_ms']:.1f})"))
# I2 LR probe
rows.append(("I2 YOLO+CLIP (LR probe)", full_probe_lr["cascade"]["mAP50"],
             full_probe_lr["cascade"]["mean_precision"], full_probe_lr["cascade"]["mean_recall"],
             fpr_probe_lr["fpr_after"],
             f"{full_probe_lr['mean_lat_ms']:.1f} ms (p95 {full_probe_lr['p95_lat_ms']:.1f})"))

print(f"{'Pipeline':<32} {'mAP50':>7} {'meanP':>7} {'meanR':>7} {'HN-FPR':>8}  Latency")
print("-" * 84)
for name, m, p, r, f, lat in rows:
    print(f"{name:<32} {m:>7.4f} {p:>7.4f} {r:>7.4f} {_pct(f):>8}  {lat}")

# Compute deltas vs I1
delta_recall = full_probe_lr["cascade"]["mean_recall"] - full_text["cascade"]["mean_recall"]
delta_fpr    = fpr_probe_lr["fpr_after"] - fpr_text["fpr_after"]
delta_map    = full_probe_lr["cascade"]["mAP50"] - full_text["cascade"]["mAP50"]
print()
print(f"I2 vs I1: ΔmAP50 = {delta_map:+.4f}  Δmean_recall = {delta_recall:+.4f}  ΔHN-FPR = {delta_fpr*100:+.3f} pp")

success_msg = []
if delta_recall > 0.01:    success_msg.append(f"recall recovered by {delta_recall*100:+.1f} pp")
if delta_fpr <= 0.005:     success_msg.append("FPR preserved or improved")
if delta_map > 0.005:      success_msg.append(f"mAP50 up {delta_map*100:+.1f} pp")
print(f"\nResult summary: {' / '.join(success_msg) if success_msg else 'no clear win — see ablations and discussion'}")


Pipeline                           mAP50   meanP   meanR   HN-FPR  Latency
------------------------------------------------------------------------------------
Baseline YOLO (yolo.val)          0.7614  0.7696  0.6971    2.44%  16.5 ms (T4)
YOLO-only (custom harness)        0.6370  0.7481  0.7272    2.44%  13.2 ms
I1 YOLO+CLIP (zero-shot)          0.5559  0.8091  0.5625    0.55%  20.8 ms (p95 42.7)
I2 YOLO+CLIP (LR probe)           0.6365  0.7649  0.7029    1.25%  21.2 ms (p95 43.7)

I2 vs I1: ΔmAP50 = +0.0806  Δmean_recall = +0.1404  ΔHN-FPR = +0.698 pp

Result summary: recall recovered by +14.0 pp / mAP50 up +8.1 pp


## 16. Save metrics JSON

Schema mirrors `improvement1_partB_metrics.json` so it can be diff'd directly in a future report. Includes I1 re-baseline numbers from this notebook (so the diff is fair) plus all I2 results and ablations.

In [26]:
summary = {
    "deliverable": "Second Improvement: YOLOv8 + CLIP Linear Probe (cascaded)",
    "dataset": "D-Fire",
    "baseline_weights": BASELINE_WEIGHTS,
    "main_configuration": {
        "clip_model":   CONFIG["clip_model"],
        "probe_type":   "logistic_regression",
        "crop_margin":  CONFIG["crop_margin"],
        "conf":         CONFIG["yolo_conf"],
        "iou":          CONFIG["yolo_iou"],
        "tau_high":     CONFIG["tau_high"],
        "mining_conf":  CONFIG["mining_conf"],
        "n_train_crops": {
            "smoke": int((y == 0).sum()),
            "fire":  int((y == 1).sum()),
            "neg":   int((y == 2).sum()),
            "neg_mined":  int(mined_features.shape[0]),
            "neg_random": int(random_features.shape[0]),
        },
    },

    # Reference: Ultralytics-native YOLO-only on test (matches baseline.ipynb)
    "yolo_only_reference_native": yolo_native,

    # I1 zero-shot, re-baselined inside this notebook on our custom harness
    "improvement1_zero_shot_reproduced": {
        "hard_negative_subset": {
            "n_images":      fpr_text["n_negatives"],
            "yolo_only_fp_imgs":  fpr_text["yolo_fp_imgs"],
            "yolo_only_fpr":      fpr_text["yolo_fpr"],
            "yolo_clip_fp_imgs":  fpr_text["fp_imgs_after"],
            "yolo_clip_fpr":      fpr_text["fpr_after"],
            "boxes_filtered":     fpr_text["boxes_filtered"],
            "n_gated":     fpr_text["n_gated"],
            "n_verified":  fpr_text["n_verified"],
            "mean_latency_ms": fpr_text["mean_latency_ms"],
        },
        "full_test_set": {
            "yolo":    full_text["yolo"],
            "cascade": full_text["cascade"],
            "n_gt":    full_text["n_gt"],
            "n_images":full_text["n_images"],
            "tau_high":full_text["tau_high"],
            "mean_lat_ms": full_text["mean_lat_ms"],
            "p95_lat_ms":  full_text["p95_lat_ms"],
        },
    },

    # I2: this notebook's contribution
    "improvement2_probe": {
        "hard_negative_subset": {
            "n_images":      fpr_probe_lr["n_negatives"],
            "yolo_only_fp_imgs":  fpr_probe_lr["yolo_fp_imgs"],
            "yolo_only_fpr":      fpr_probe_lr["yolo_fpr"],
            "yolo_clip_fp_imgs":  fpr_probe_lr["fp_imgs_after"],
            "yolo_clip_fpr":      fpr_probe_lr["fpr_after"],
            "boxes_filtered":     fpr_probe_lr["boxes_filtered"],
            "n_gated":     fpr_probe_lr["n_gated"],
            "n_verified":  fpr_probe_lr["n_verified"],
            "mean_latency_ms": fpr_probe_lr["mean_latency_ms"],
        },
        "full_test_set": {
            "yolo":    full_probe_lr["yolo"],
            "cascade": full_probe_lr["cascade"],
            "n_gt":    full_probe_lr["n_gt"],
            "n_images":full_probe_lr["n_images"],
            "tau_high":full_probe_lr["tau_high"],
            "mean_lat_ms": full_probe_lr["mean_lat_ms"],
            "p95_lat_ms":  full_probe_lr["p95_lat_ms"],
        },
    },

    "ablation_probe_type":      abl_probe_type,
    "ablation_cascade_threshold": abl_tau,
    "ablation_crop_margin":     abl_margin,
    "ablation_neg_mining":      abl_neg_mining,

    "latency_bench": latency_bench,

    "comparison_table": {
        "baseline_native":      {"mAP50": yolo_native["mAP50"],
                                  "mean_precision": yolo_native["mean_precision"],
                                  "mean_recall": yolo_native["mean_recall"]},
        "yolo_only_custom":     full_text["yolo"],
        "i1_zero_shot":         full_text["cascade"],
        "i2_lr_probe":          full_probe_lr["cascade"],
    },

    "deltas_i2_vs_i1": {
        "delta_mAP50":         full_probe_lr["cascade"]["mAP50"] - full_text["cascade"]["mAP50"],
        "delta_mean_recall":   full_probe_lr["cascade"]["mean_recall"] - full_text["cascade"]["mean_recall"],
        "delta_mean_precision":full_probe_lr["cascade"]["mean_precision"] - full_text["cascade"]["mean_precision"],
        "delta_hn_fpr":        fpr_probe_lr["fpr_after"] - fpr_text["fpr_after"],
    },
}

with open(CONFIG["metrics_path"], "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"Saved: {CONFIG['metrics_path']}")
print(f"\nKey deltas (I2 vs I1):")
for k, v in summary["deltas_i2_vs_i1"].items():
    print(f"  {k:25s} = {v:+.4f}")


Saved: /kaggle/working/improvement2_metrics.json

Key deltas (I2 vs I1):
  delta_mAP50               = +0.0806
  delta_mean_recall         = +0.1404
  delta_mean_precision      = -0.0442
  delta_hn_fpr              = +0.0070


In [27]:

  # === Patch 1: more aggressive hard-negative mining ===
  NEW_MINING_CONF = 0.03

  def _collect_negs(img_dir, lbl_dir):
      out = []
      for fname in os.listdir(lbl_dir):
          if not fname.endswith('.txt'): continue
          lp = os.path.join(lbl_dir, fname)
          if os.path.getsize(lp) > 0: continue
          stem = os.path.splitext(fname)[0]
          for ext in ('.jpg', '.jpeg', '.png'):
              cand = os.path.join(img_dir, stem + ext)
              if os.path.exists(cand): out.append(cand); break
      return out

  train_negs2 = _collect_negs(train_img, train_lbl)
  val_negs2   = _collect_negs(val_img,   val_lbl)
  all_negs2 = train_negs2 + val_negs2
  print(f"Mining on {len(train_negs2)} train + {len(val_negs2)} val "
        f"= {len(all_negs2)} negatives at conf={NEW_MINING_CONF}")

  mined2_crops = []
  for path in tqdm(all_negs2, desc="mine v2", unit="img"):
      pred = yolo.predict(source=path, imgsz=CONFIG["imgsz"], conf=NEW_MINING_CONF,
                          iou=CONFIG["yolo_iou"], device=device, verbose=False)[0]
      if len(pred.boxes) == 0: continue
      try:
          pil = Image.open(path).convert("RGB")
      except Exception:
          continue
      for b in pred.boxes:
          x1, y1, x2, y2 = b.xyxy[0].cpu().numpy()
          crop = extract_crop(pil, (x1, y1, x2, y2), margin=CONFIG["crop_margin"])
          if crop.size[0] < 4 or crop.size[1] < 4: continue
          mined2_crops.append(crop)

  print(f"\nMined v2 crops: {len(mined2_crops)}  (was {mined_features.shape[0]})")
  print("Encoding...")
  mined2_features = encode_crops_batched(mined2_crops, batch_size=128)
  print(f"Mined v2 features: {mined2_features.shape}")
  del mined2_crops


Mining on 6458 train + 1375 val = 7833 negatives at conf=0.03


mine v2: 100%|██████████| 7833/7833 [02:03<00:00, 63.21img/s]



Mined v2 crops: 1071  (was 282)
Encoding...
Mined v2 features: (1071, 512)


In [28]:
  # === Patch 2: text-anchored negative pseudo-features ===
  # I1's 12 confounder text embeddings are already in CLIP feature space — bake them
  # into the probe by using them as extra negative training points (with small noise so
  # the probe learns a region around each, not a single point).

  neg_text_anchors = text_feats[N_FIRE + N_SMOKE:].float().cpu().numpy()  # (12, 512), L2-normed
  print(f"I1 confounder anchors: {neg_text_anchors.shape}")

  TEXT_ANCHOR_REPLICAS = 50
  TEXT_ANCHOR_NOISE    = 0.02

  rng = np.random.default_rng(SEED)
  samples = []
  for v in neg_text_anchors:
      s = v[None, :] + rng.normal(0, TEXT_ANCHOR_NOISE, size=(TEXT_ANCHOR_REPLICAS, v.shape[0]))
      s = s / np.linalg.norm(s, axis=1, keepdims=True)
      samples.append(s.astype(np.float32))
  text_anchor_features = np.concatenate(samples, axis=0)
  print(f"Text-anchored synthetic negatives: {text_anchor_features.shape}")


I1 confounder anchors: (12, 512)
Text-anchored synthetic negatives: (600, 512)


In [29]:
# === Patch 3: rebuild train set, drop most random negatives, retrain ===
CAP_RANDOM_NEG = 500   # was 5000 — stop drowning the mined+anchored signal
random_features_capped = random_features[:CAP_RANDOM_NEG]

X2_neg = np.concatenate([
  mined2_features,         # 700-1500 hard mined negs (aggressive conf + val)
  random_features_capped,  # 500 generic
  text_anchor_features,    # 600 text-anchored synthetic
], axis=0).astype(np.float32)

X2 = np.concatenate([pos_features, X2_neg], axis=0).astype(np.float32)
y2 = np.array(pos_labels + [2] * X2_neg.shape[0], dtype=np.int64)

perm = np.random.default_rng(SEED).permutation(len(y2))
X2, y2 = X2[perm], y2[perm]
print(f"Training set v2: smoke={int((y2==0).sum())} fire={int((y2==1).sum())} "
    f"neg={int((y2==2).sum())}  (mined2={mined2_features.shape[0]}, "
    f"random_cap={CAP_RANDOM_NEG}, text_anchor={text_anchor_features.shape[0]})  total={len(y2)}")

idx_tr2, idx_va2 = stratified_split(X2, y2, val_frac=0.1, seed=SEED)

probe_lr_v2 = LogisticRegression(
  C=CONFIG["lr_C"], max_iter=CONFIG["lr_max_iter"],
  multi_class="multinomial", class_weight="balanced",
  solver="lbfgs", random_state=SEED,
)
probe_lr_v2.fit(X2[idx_tr2], y2[idx_tr2])

print("\nProbe v2 — held-out 10% report:")
print(classification_report(y2[idx_va2], probe_lr_v2.predict(X2[idx_va2]),
                          target_names=PROBE_CLASS_NAMES, digits=4))

verify_probe_lr_v2 = make_clip_verify_probe_lr(probe_lr_v2)


Training set v2: smoke=7788 fire=9629 neg=2171  (mined2=1071, random_cap=500, text_anchor=600)  total=19588

Probe v2 — held-out 10% report:
              precision    recall  f1-score   support

       smoke     0.9043    0.8136    0.8566       778
        fire     0.9392    0.9314    0.9353       962
         neg     0.6304    0.8802    0.7346       217

    accuracy                         0.8789      1957
   macro avg     0.8246    0.8751    0.8422      1957
weighted avg     0.8911    0.8789    0.8817      1957



In [30]:
  # === Patch 4: hard-neg FPR + full test eval for v2 probe ===
  print("=== Hard-neg FPR: probe v2 ===")
  fpr_probe_v2 = evaluate_fpr_on_negatives(neg_test_imgs, verify_probe_lr_v2, desc="probe v2")
  print(json.dumps(fpr_probe_v2, indent=2))

  print("\n=== Full test eval: probe v2 ===")
  full_probe_v2 = full_test_eval(test_img, test_lbl, verify_probe_lr_v2,
                                 max_images=CONFIG["max_eval_images"], desc="probe v2")
  print(json.dumps({k: v for k, v in full_probe_v2.items() if k != "n_gt"}, indent=2))


=== Hard-neg FPR: probe v2 ===


probe v2: 100%|██████████| 2005/2005 [00:29<00:00, 66.85img/s]


{
  "n_negatives": 2005,
  "yolo_fp_imgs": 49,
  "yolo_fp_boxes": 55,
  "yolo_fpr": 0.024438902743142144,
  "fp_imgs_after": 14,
  "fp_boxes_after": 16,
  "fpr_after": 0.006982543640897756,
  "boxes_filtered": 39,
  "n_gated": 2,
  "n_verified": 53,
  "mean_latency_ms": 14.783487141647768
}

=== Full test eval: probe v2 ===


full eval probe v2: 100%|██████████| 4306/4306 [01:48<00:00, 39.81img/s]

{
  "yolo": {
    "ap50_smoke": 0.6948293172257434,
    "precision_smoke": 0.8074041427941825,
    "recall_smoke": 0.7913606911447084,
    "ap50_fire": 0.5791710207045309,
    "precision_fire": 0.6888086642599278,
    "recall_fire": 0.6629603891591382,
    "mAP50": 0.6370001689651371,
    "mean_precision": 0.7481064035270552,
    "mean_recall": 0.7271605401519233
  },
  "cascade": {
    "ap50_smoke": 0.6939603457764402,
    "precision_smoke": 0.8449062341611758,
    "recall_smoke": 0.7200863930885529,
    "ap50_fire": 0.5770161102047524,
    "precision_fire": 0.6978228228228228,
    "recall_fire": 0.6459346768589298,
    "mAP50": 0.6354882279905962,
    "mean_precision": 0.7713645284919993,
    "mean_recall": 0.6830105349737414
  },
  "n_images": 4306,
  "tau_high": 0.7,
  "mean_lat_ms": 21.11446215560053,
  "p95_lat_ms": 42.63134050006556
}


In [31]:
# === Patch 5: ensemble verifier ===
# Reject a detection if EITHER the LR probe v2 OR the I1 text-prompt argmax says "neg".
# Otherwise trust the probe's class call. Best of both: I1's strong rejection prior +
# probe's recall preservation.

def make_clip_verify_ensemble(probe):
  @torch.no_grad()
  def _verify(image_pil, boxes_xyxy, crop_margin):
      feats_t = encode_detection_crops(image_pil, boxes_xyxy, crop_margin)
      if feats_t.shape[0] == 0: return []
      sims    = (feats_t @ text_feats.T).cpu().numpy()
      feats_np = feats_t.float().cpu().numpy()
      proba    = probe.predict_proba(feats_np)
      out = []
      for s_row, p_row in zip(sims, proba):
          text_b  = bucket_of_text(int(np.argmax(s_row)))
          probe_b = PROBE_CLASS_NAMES[int(np.argmax(p_row))]
          top_b = "neg" if (text_b == "neg" or probe_b == "neg") else probe_b
          out.append({
              "top_idx": int(np.argmax(p_row)), "top_bucket": top_b,
              "top_score": float(p_row.max()),
              "smoke_score": float(p_row[0]), "fire_score": float(p_row[1]),
              "neg_score":   float(p_row[2]),
          })
      return out
  return _verify

verify_ensemble = make_clip_verify_ensemble(probe_lr_v2)

print("=== Hard-neg FPR: ensemble ===")
fpr_ens = evaluate_fpr_on_negatives(neg_test_imgs, verify_ensemble, desc="ensemble")
print(json.dumps(fpr_ens, indent=2))

print("\n=== Full test eval: ensemble ===")
full_ens = full_test_eval(test_img, test_lbl, verify_ensemble,
                        max_images=CONFIG["max_eval_images"], desc="ensemble")
print(json.dumps({k: v for k, v in full_ens.items() if k != "n_gt"}, indent=2))


=== Hard-neg FPR: ensemble ===


ensemble: 100%|██████████| 2005/2005 [00:26<00:00, 75.24img/s]


{
  "n_negatives": 2005,
  "yolo_fp_imgs": 49,
  "yolo_fp_boxes": 55,
  "yolo_fpr": 0.024438902743142144,
  "fp_imgs_after": 6,
  "fp_boxes_after": 6,
  "fpr_after": 0.0029925187032418953,
  "boxes_filtered": 49,
  "n_gated": 2,
  "n_verified": 53,
  "mean_latency_ms": 13.148914051863922
}

=== Full test eval: ensemble ===


full eval ensemble: 100%|██████████| 4306/4306 [01:46<00:00, 40.44img/s]

{
  "yolo": {
    "ap50_smoke": 0.6948293172257434,
    "precision_smoke": 0.8074041427941825,
    "recall_smoke": 0.7913606911447084,
    "ap50_fire": 0.5791710207045309,
    "precision_fire": 0.6888086642599278,
    "recall_fire": 0.6629603891591382,
    "mAP50": 0.6370001689651371,
    "mean_precision": 0.7481064035270552,
    "mean_recall": 0.7271605401519233
  },
  "cascade": {
    "ap50_smoke": 0.6146251028344313,
    "precision_smoke": 0.8524240809802877,
    "recall_smoke": 0.6911447084233261,
    "ap50_fire": 0.5004877806197616,
    "precision_fire": 0.7412345679012345,
    "recall_fire": 0.5215427380125087,
    "mAP50": 0.5575564417270964,
    "mean_precision": 0.7968293244407612,
    "mean_recall": 0.6063437232179174
  },
  "n_images": 4306,
  "tau_high": 0.7,
  "mean_lat_ms": 21.028798944263652,
  "p95_lat_ms": 43.195227749833975
}


In [32]:
with open(CONFIG["metrics_path"]) as f:
  summary = json.load(f)

summary["improvement2_probe_v2"] = {
  "config": {
      "mining_conf": NEW_MINING_CONF, "mined_on": "train+val",
      "cap_random_neg": CAP_RANDOM_NEG,
      "text_anchor_replicas": TEXT_ANCHOR_REPLICAS,
      "text_anchor_noise": TEXT_ANCHOR_NOISE,
      "n_train_crops": {
          "smoke": int((y2 == 0).sum()), "fire": int((y2 == 1).sum()),
          "neg":   int((y2 == 2).sum()),
          "neg_mined":         int(mined2_features.shape[0]),
          "neg_random_capped": int(min(CAP_RANDOM_NEG, random_features.shape[0])),
          "neg_text_anchored": int(text_anchor_features.shape[0]),
      },
  },
  "hard_negative_subset": {
      "n_images":         fpr_probe_v2["n_negatives"],
      "yolo_only_fpr":    fpr_probe_v2["yolo_fpr"],
      "yolo_clip_fpr":    fpr_probe_v2["fpr_after"],
      "yolo_clip_fp_imgs":fpr_probe_v2["fp_imgs_after"],
      "boxes_filtered":   fpr_probe_v2["boxes_filtered"],
      "mean_latency_ms":  fpr_probe_v2["mean_latency_ms"],
  },
  "full_test_set": {
      "yolo": full_probe_v2["yolo"], "cascade": full_probe_v2["cascade"],
      "n_images": full_probe_v2["n_images"], "tau_high": full_probe_v2["tau_high"],
      "mean_lat_ms": full_probe_v2["mean_lat_ms"], "p95_lat_ms": full_probe_v2["p95_lat_ms"],
  },
}
summary["improvement2_ensemble"] = {
  "description": "probe_v2 OR I1 text-prompt rejects",
  "hard_negative_subset": {
      "n_images":         fpr_ens["n_negatives"],
      "yolo_only_fpr":    fpr_ens["yolo_fpr"],
      "yolo_clip_fpr":    fpr_ens["fpr_after"],
      "yolo_clip_fp_imgs":fpr_ens["fp_imgs_after"],
      "boxes_filtered":   fpr_ens["boxes_filtered"],
      "mean_latency_ms":  fpr_ens["mean_latency_ms"],
  },
  "full_test_set": {
      "yolo": full_ens["yolo"], "cascade": full_ens["cascade"],
      "n_images": full_ens["n_images"], "tau_high": full_ens["tau_high"],
      "mean_lat_ms": full_ens["mean_lat_ms"], "p95_lat_ms": full_ens["p95_lat_ms"],
  },
}

with open(CONFIG["metrics_path"], "w") as f:
  json.dump(summary, f, indent=2, default=str)

# Final comparison
i1   = summary["improvement1_zero_shot_reproduced"]
i2v1 = summary["improvement2_probe"]
print(f"\n{'Method':<28} {'HN-FPR':>8} {'mAP50':>7} {'meanP':>7} {'meanR':>7}")
print("-" * 64)
def _row(name, fpr, full):
  print(f"{name:<28} {fpr*100:>7.2f}% {full['mAP50']:>7.4f} "
        f"{full['mean_precision']:>7.4f} {full['mean_recall']:>7.4f}")
_row("YOLO-only (custom)", i1["hard_negative_subset"]["yolo_only_fpr"], i1["full_test_set"]["yolo"])
_row("I1 zero-shot",       i1["hard_negative_subset"]["yolo_clip_fpr"], i1["full_test_set"]["cascade"])
_row("I2 probe v1 (orig)", i2v1["hard_negative_subset"]["yolo_clip_fpr"], i2v1["full_test_set"]["cascade"])
_row("I2 probe v2",        fpr_probe_v2["fpr_after"], full_probe_v2["cascade"])
_row("I2 ensemble",        fpr_ens["fpr_after"], full_ens["cascade"])
print(f"\nUpdated: {CONFIG['metrics_path']}")



Method                         HN-FPR   mAP50   meanP   meanR
----------------------------------------------------------------
YOLO-only (custom)              2.44%  0.6370  0.7481  0.7272
I1 zero-shot                    0.55%  0.5559  0.8091  0.5625
I2 probe v1 (orig)              1.25%  0.6365  0.7649  0.7029
I2 probe v2                     0.70%  0.6355  0.7714  0.6830
I2 ensemble                     0.30%  0.5576  0.7968  0.6063

Updated: /kaggle/working/improvement2_metrics.json


## 17. Discussion

Fill this in after the run produces real numbers. Skeleton:

### 17.1 Did the probe recover recall?
- Compare `delta_mean_recall` (I2 − I1). The hypothesis predicted +5 to +10 pp; check what actually happened, broken down by class.
- If recall did not recover, the most likely culprits are: (a) probe overfit to mined negatives (look at the held-out 10% acc — if `neg` precision is much higher than smoke/fire recall, this is the signal), (b) crop distribution at inference time differs from training (training crops come from GT boxes; inference crops come from low-confidence YOLO boxes), (c) probe is well-calibrated on training but the cascade's auto-keep at τ_high=0.7 is doing most of the heavy lifting either way.

### 17.2 Did the FPR survive?
- I2's hard-neg FPR should be in the range 0.4–0.7%. Anything materially higher than I1's 0.55% is a regression and should be discussed honestly. If higher: try lowering `mining_conf` further (catch more confounders during training) or rebalancing the training set.

### 17.3 What the ablations tell us
- **Probe type (LR vs MLP):** if MLP is meaningfully better, our 512-d feature space is *not* linearly separable for this task — interesting for the report.
- **τ_high sweep:** should be qualitatively similar to I1's monotone curve. If the I2 curve is *flatter* near τ=0.7, that suggests the probe is more reliable than text prompts at low YOLO confidences — a clear win story.
- **Crop margin:** shape should match I1 (best at 0–40%, worse at 60%). If it doesn't, the probe is sensitive in ways the text prompts weren't — flag it.
- **Negative mining:** mined-only should be slightly worse than mined+random (less coverage of generic negatives). If it's *better*, that's a striking finding — random crops are adding label noise.

### 17.4 What to flag as honest limitations
1. The probe trains on D-Fire train; it has not seen the WWF PTZ camera distribution. Real deployment will need fine-tuning on PTZ data once that's collected.
2. Mining at conf=0.10 is somewhat arbitrary. A better strategy would be to mine at the τ_high threshold (since that's exactly the regime the probe sees at inference), but mining is one-shot and cheap, so we use a wider net.
3. The 12-prompt zero-shot bank in I1 was a strong baseline because the prompts were hand-engineered for this domain. The probe replaces engineered priors with learned ones; the comparison is fair *given those priors* but doesn't tell us what would happen with a different prompt bank.
